In [1]:
# =============================================================================
# STEP 2 - THRESHOLD-SPECIFIC DISPLACEMENT
#
# The manuscript's mechanism is stated with a two-sided Kolmogorov-Smirnov
# distance, which is the largest gap between the two distribution functions
# ANYWHERE on the range. Coverage does not depend on that. It depends on the
# target distribution evaluated at one point, the source-calibrated threshold:
#
#     coverage_t,c = F_t,c(q_s,c)
#
# so the deficit decomposes as
#
#     (1-a) - F_t,c(q_s,c)  =  [F_s,c(q_s,c) - F_t,c(q_s,c)]  +  [(1-a) - F_s,c(q_s,c)]
#                               signed displacement AT q_s        calibration slack, <= 1/(n+1)
#
# NOTE ON CIRCULARITY. The left-hand side IS the undercoverage by definition, so
# regressing it on undercoverage proves nothing and is not done here. What this
# notebook measures is the RELATIONSHIP BETWEEN THE PROXY AND THE EXACT TERM:
# how much of the KS movement is realised at the threshold, whether movement is
# harmful or protective in direction, and whether KS ever badly overstates harm.
# Two distributions can share a KS of 0.5 while one moves entirely below the
# threshold and the other entirely above; KS cannot tell them apart.
# =============================================================================
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, shutil, glob, subprocess, hashlib
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
import importlib
for m in ['config','conformal']:
    if m in sys.modules: importlib.reload(sys.modules[m])
import config, conformal
from conformal import conformal_q
import numpy as np, pandas as pd
from scipy import stats
RD=config.REPORTS_DIR; ALPHA=config.ALPHA_PRIMARY
print('ready | alpha', ALPHA)


Mounted at /content/drive
ready | alpha 0.05


In [2]:
# =============================================================================
# Cell 2 - the measures. Deterministic mid-point APS, matching the convention
# used by the mechanism analyses so these rows are comparable with them.
# =============================================================================
def aps_mid(P):
    o=np.argsort(-P,axis=1); sp=np.take_along_axis(P,o,1); cum=np.cumsum(sp,1)
    ss=cum-0.5*sp; out=np.empty_like(P); np.put_along_axis(out,o,ss,1); return out

def ecdf_at(sample, t):
    """F(t) for a sample, evaluated at a scalar threshold."""
    s=np.sort(np.asarray(sample,float))
    return float(np.searchsorted(s, t, side='right')/max(len(s),1))

def displacement(src, tgt, alpha):
    """All quantities on one (class, model) cell."""
    src=np.asarray(src,float); tgt=np.asarray(tgt,float)
    if len(src)<5 or len(tgt)<5: return None
    q,_=conformal_q(src, alpha)
    if not np.isfinite(q): return None
    Fs_q=ecdf_at(src,q); Ft_q=ecdf_at(tgt,q)
    # two-sided KS and where it occurs
    grid=np.unique(np.concatenate([src,tgt]))
    Fs=np.searchsorted(np.sort(src),grid,side='right')/len(src)
    Ft=np.searchsorted(np.sort(tgt),grid,side='right')/len(tgt)
    diff=Fs-Ft                                  # positive = target mass has moved UP (harmful)
    ks=float(np.max(np.abs(diff)))
    ks_at=float(grid[int(np.argmax(np.abs(diff)))])
    one_sided_up=float(np.max(diff)); one_sided_down=float(np.max(-diff))
    return {'q_src':float(q),'F_src_at_q':Fs_q,'F_tgt_at_q':Ft_q,
            'signed_displacement_at_q':float(Fs_q-Ft_q),      # the exact harmful term
            'calibration_slack':float((1-alpha)-Fs_q),        # bounded by 1/(n+1)
            'undercoverage':float((1-alpha)-Ft_q),
            'KS_two_sided':ks,'KS_location':ks_at,
            'one_sided_up':one_sided_up,'one_sided_down':one_sided_down,
            'tail_mass_above_q_target':float(1-Ft_q),
            'quantile_displacement':float(np.quantile(tgt,1-alpha)-q),
            'frac_of_KS_realised_at_q':float(abs(Fs_q-Ft_q)/ks) if ks>0 else np.nan,
            'n_src':int(len(src)),'n_tgt':int(len(tgt))}
print('measures defined')
print('  signed_displacement_at_q > 0 means target mass moved ABOVE the source threshold (harmful)')
print('  frac_of_KS_realised_at_q near 1 means the proxy and the exact term agree;')
print('  near 0 means KS is measuring movement that does not touch the threshold')


measures defined
  signed_displacement_at_q > 0 means target mass moved ABOVE the source threshold (harmful)
  frac_of_KS_realised_at_q near 1 means the proxy and the exact term agree;
  near 0 means KS is measuring movement that does not touch the threshold


In [3]:
# =============================================================================
# Cell 3 - NSL-KDD (rung 0.80) and UGR'16, from cached probabilities.
# =============================================================================
rows=[]

# ---------- NSL-KDD ----------
CL=config.CANONICAL_CLASSES; c2i={c:i for i,c in enumerate(CL)}; K=len(CL)
tr=pd.read_parquet(config.INTERIM_DIR/'nslkdd_train.parquet').reset_index(drop=True)
te=pd.read_parquet(config.INTERIM_DIR/'nslkdd_test.parquet').reset_index(drop=True)
part=pd.read_parquet(config.PROC_DIR/'nslkdd_source_partition_labels.parquet')
tr=tr.assign(partition=part['partition'].values)
y_sp=tr[tr.partition=='source_cal_pool']['label'].map(c2i).to_numpy()
y_te=te['label'].map(c2i).to_numpy()
assign=pd.read_parquet(config.PROC_DIR/'nslkdd_ladder_assignments.parquet')
IDX={(r,j,role):g['test_idx'].to_numpy() for (r,j,role),g in assign.groupby(['rung','realization','role'])}
REALS=sorted(assign[np.isclose(assign.rung,0.80)]['realization'].unique())
for f in sorted(glob.glob(str(config.PROC_DIR/'probs_*.npz'))):
    arch,seed=Path(f).stem.replace('probs_','').rsplit('_s',1)
    d=np.load(f); S_sp=aps_mid(d['S_pool'].astype(np.float64)); P_te=d['target'].astype(np.float64)
    for j in REALS:
        ev=IDX.get((0.80,j,'eval'))
        if ev is None or len(ev)==0: continue
        S_ev=aps_mid(P_te[ev]); y_ev=y_te[ev]
        for k in range(K):
            r=displacement(S_sp[y_sp==k,k], S_ev[y_ev==k,k], ALPHA)
            if r: rows.append({'dataset':'nslkdd','rung':0.80,'class':CL[k],'arch':arch,
                               'seed':seed,'realization':int(j),**r})
print('NSL rows:', sum(1 for r in rows if r['dataset']=='nslkdd'))

# ---------- UGR'16 ----------
UGR=config.DATASETS_DIR/'ugr16'
us=pd.read_parquet(UGR/'july_week5.parquet'); ut=pd.read_parquet(UGR/'august_week1.parquet')
for dd in (us,ut): dd['label']=dd['label'].astype(str).str.strip().str.lower()
UK=['background','dos','scan11','scan44','nerisbotnet']
us=us[us.label.isin(UK)].reset_index(drop=True); ut=ut[ut.label.isin(UK)].reset_index(drop=True)
UCL=sorted(UK); U2I={c:i for i,c in enumerate(UCL)}
def strat(df,fr,seed,col='label'):
    rg=np.random.default_rng(seed); nm=list(fr); ff=np.array([fr[k] for k in nm],float); big=nm[int(np.argmax(ff))]
    a=pd.Series(index=df.index,dtype=object)
    for _,s in df.groupby(col,sort=True):
        idx=s.index.to_numpy().copy(); rg.shuffle(idx); n=len(idx)
        c=np.floor(ff*n).astype(int); c[nm.index(big)]+=n-c.sum(); kk=0
        for a2,q in zip(nm,c): a.loc[idx[kk:kk+q]]=a2; kk+=q
    return a
us=us.assign(partition=strat(us,config.SPLIT_FRACTIONS,20260725).values)
y_usp=us[us.partition=='source_cal_pool']['label'].map(U2I).to_numpy()
y_utg=ut['label'].map(U2I).to_numpy()
for f in sorted((config.DATA_DIR/'ugr16_probs').glob('ugr16__*.npz')):
    _,arch,sd=Path(f).stem.split('__'); seed=int(sd.replace('seed',''))
    d=np.load(f); S_sp=aps_mid(d['srcpool'].astype(np.float64)); S_tg=aps_mid(d['target'].astype(np.float64))
    for k in range(len(UCL)):
        r=displacement(S_sp[y_usp==k,k], S_tg[y_utg==k,k], ALPHA)
        if r: rows.append({'dataset':'ugr16','rung':np.nan,'class':UCL[k],'arch':arch,
                           'seed':str(seed),'realization':0,**r})
print('UGR rows:', sum(1 for r in rows if r['dataset']=='ugr16'))


NSL rows: 2400
UGR rows: 150


In [4]:
# =============================================================================
# Cell 4 - CIC-IoT-2023 (rung 0.80), the environment where nothing fails. This is
# the discriminating case: if KS is small AND the threshold displacement is small,
# the proxy and the exact term agree that there is no harm.
# =============================================================================
iot=pd.read_parquet(config.PROC_DIR/'ciciot2023_prepared.parquet')
sp=pd.read_parquet(config.PROC_DIR/'ciciot2023_split.parquet')
iot['side']=sp['side'].values; iot['partition']=sp['partition'].values
lad=pd.read_parquet(config.PROC_DIR/'ciciot2023_ladder_assignments.parquet')
mrec=json.loads((RD/'ciciot2023_model_record.json').read_text())
ICL=mrec['classes_canonical_order']; ic2i={c:i for i,c in enumerate(ICL)}
iot['y']=iot['family'].map(ic2i).astype(np.int64)
sc_idx=iot.index[iot.partition=='source_cal_pool'].to_numpy()
tg_idx=iot.index[iot.partition=='target_pool'].to_numpy()
TGT=np.full(len(iot),-1,dtype=np.int64); TGT[tg_idx]=np.arange(len(tg_idx))
YV=iot['y'].to_numpy(); y_isp=YV[sc_idx]
top=lad['rung'].max()
for f in sorted((config.DATA_DIR/'ciciot_probs').glob('ciciot2023__*.npz')):
    _,arch,sd=Path(f).stem.split('__'); seed=int(sd.replace('seed',''))
    d=np.load(f); S_sp=aps_mid(d['srcpool'].astype(np.float64)); P_tg=d['target']
    for j,g in lad[np.isclose(lad.rung,top)].groupby('realization'):
        ev=g['row_idx'].to_numpy(); S_ev=aps_mid(P_tg[TGT[ev]].astype(np.float64)); y_ev=YV[ev]
        for k in range(len(ICL)):
            r=displacement(S_sp[y_isp==k,k], S_ev[y_ev==k,k], ALPHA)
            if r: rows.append({'dataset':'ciciot2023','rung':float(top),'class':ICL[k],
                               'arch':arch,'seed':str(seed),'realization':int(j),**r})
D=pd.DataFrame(rows)
print('total rows:', len(D), '| per dataset:', D.groupby('dataset').size().to_dict())


total rows: 3750 | per dataset: {'ciciot2023': 1200, 'nslkdd': 2400, 'ugr16': 150}


In [5]:
# =============================================================================
# Cell 5 - the three non-circular questions.
# =============================================================================
cl=D.groupby(['dataset','class'],as_index=False)[
   ['signed_displacement_at_q','calibration_slack','undercoverage','KS_two_sided',
    'one_sided_up','one_sided_down','tail_mass_above_q_target','quantile_displacement',
    'frac_of_KS_realised_at_q']].mean()

print('Q1. HOW MUCH OF THE KS MOVEMENT IS REALISED AT THE THRESHOLD?')
print(cl[['dataset','class','KS_two_sided','signed_displacement_at_q','frac_of_KS_realised_at_q']]
      .round(4).to_string(index=False))

print('\nQ2. IS THE MOVEMENT HARMFUL OR PROTECTIVE IN DIRECTION?')
cl['direction']=np.where(cl.signed_displacement_at_q>0.01,'harmful',
                  np.where(cl.signed_displacement_at_q<-0.01,'protective','neutral'))
print(cl.groupby(['dataset','direction']).size().to_string())
print('\n  one-sided components (up = target mass above the threshold = harmful):')
print(cl[['dataset','class','one_sided_up','one_sided_down']].round(4).to_string(index=False))

print('\nQ3. DOES KS EVER BADLY OVERSTATE HARM?')
over=cl[(cl.KS_two_sided>0.2)&(cl.signed_displacement_at_q.abs()<0.05)]
print('  classes with large KS but negligible threshold displacement:')
print(over[['dataset','class','KS_two_sided','signed_displacement_at_q']].round(4).to_string(index=False)
      if len(over) else '   none')

print('\nIDENTITY CHECK (must hold by construction, validates the implementation):')
cl['reconstructed']=cl.signed_displacement_at_q+cl.calibration_slack
err=(cl.reconstructed-cl.undercoverage).abs().max()
print(f'  max |(signed displacement + calibration slack) - undercoverage| = {err:.2e}')
print('  calibration slack range:', round(cl.calibration_slack.min(),5), 'to', round(cl.calibration_slack.max(),5))

r_ks,p_ks=stats.spearmanr(cl.KS_two_sided, cl.undercoverage)
r_up,p_up=stats.spearmanr(cl.one_sided_up, cl.undercoverage)
print(f'\nPROXY QUALITY (not a test of the mechanism, a test of the PROXY):')
print(f'  two-sided KS   vs undercoverage: rho={r_ks:.3f} (p={p_ks:.2g})')
print(f'  one-sided up   vs undercoverage: rho={r_up:.3f} (p={p_up:.2g})')
print('  the one-sided statistic is directional and should track more closely;')
print('  if it does, the paper should report it in place of the two-sided KS.')


Q1. HOW MUCH OF THE KS MOVEMENT IS REALISED AT THE THRESHOLD?
   dataset       class  KS_two_sided  signed_displacement_at_q  frac_of_KS_realised_at_q
ciciot2023      Benign        0.0180                    0.0009                    0.1825
ciciot2023  BruteForce        0.0394                   -0.0044                    0.2243
ciciot2023        DDoS        0.0022                   -0.0000                    0.2069
ciciot2023         DoS        0.0044                    0.0002                    0.2314
ciciot2023       Mirai        0.0022                   -0.0003                    0.2723
ciciot2023       Recon        0.0090                   -0.0015                    0.2845
ciciot2023    Spoofing        0.0156                   -0.0025                    0.2767
ciciot2023         Web        0.0893                    0.0001                    0.1208
    nslkdd         DoS        0.6730                    0.6716                    0.9979
    nslkdd      Normal        0.0496            

In [6]:
# =============================================================================
# Cell 6 - save and commit
# =============================================================================
D.to_csv(RD/'threshold_displacement_cells.csv', index=False)
cl.to_csv(RD/'threshold_displacement_class_level.csv', index=False)
(RD/'threshold_displacement_verdict.json').write_text(json.dumps({
 'purpose':'separate the exact threshold-specific term from the two-sided KS proxy',
 'identity':'undercoverage = [F_s(q_s) - F_t(q_s)] + [(1-a) - F_s(q_s)]',
 'circularity_note':'the left-hand side is undercoverage by definition; this notebook does '
                    'NOT regress it on undercoverage. It quantifies how much of the KS '
                    'movement is realised at the threshold, the direction of movement, and '
                    'whether KS overstates harm.',
 'max_identity_error':float((cl.signed_displacement_at_q+cl.calibration_slack-cl.undercoverage).abs().max()),
 'proxy_quality':{'two_sided_KS_rho':float(stats.spearmanr(cl.KS_two_sided,cl.undercoverage)[0]),
                  'one_sided_up_rho':float(stats.spearmanr(cl.one_sided_up,cl.undercoverage)[0])},
 'class_level':cl.round(5).to_dict('records')}, indent=2, default=str))
print('saved threshold_displacement_{cells,class_level}.csv and the verdict')

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT); git('add','-A',show=False)
if git('status','--porcelain',show=False).stdout.strip():
    git('commit','-m','step 2: threshold-specific displacement; separates the exact harmful term from the two-sided KS proxy')
    r=git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)


saved threshold_displacement_{cells,class_level}.csv and the verdict
[main b5ca85d] step 2: threshold-specific displacement; separates the exact harmful term from the two-sided KS proxy
 5 files changed, 4038 insertions(+), 1 deletion(-)
 create mode 100644 notebooks/37_threshold_displacement.ipynb
 create mode 100644 reports/threshold_displacement_cells.csv
 create mode 100644 reports/threshold_displacement_class_level.csv
 create mode 100644 reports/threshold_displacement_verdict.json
Branch 'main' set up to track remote branch 'main' from 'origin'.
To https://github.com/anasbiswas1/calshift-research.git
   7b1cf19..b5ca85d  main -> main
b5ca85d step 2: threshold-specific displacement; separates the exact harmful term from the two-sided KS proxy
7b1cf19 step 1: immutable results ledger (final_results.json) regenerated from committed reports, plus manuscript number checker
2bf79c0 rebalance mechanism table to one rung per laddered dataset; pooled mechanism recomputed across four datas